# Procesamiento de datos en Python: práctica de análisis exploratorio (EDA)
### Misión Académica «Inteligencia Empresarial en la Era de los Datos» · Universidad Andina del Cusco

**Versión estudiante**

Este cuaderno es la parte práctica de la clase. Aquí vas a aplicar, con una base de datos real, las preguntas que vimos en las diapositivas:

| Sección | Pregunta | Diapositivas |
|---|---|---|
| 1 | ¿Qué es una fila y qué significa cada columna? | 3, 18 |
| 2 | ¿Qué tipo de variable es cada columna? | 4 |
| 3 | ¿Cuántas observaciones y variables hay? ¿Hay redundancia? | 6–7 |
| 4 | ¿Cómo se distribuye cada variable? | 9–11 |
| 5 | ¿Dónde falta información y qué decidimos? | 12 |
| 6 | ¿Hay repeticiones o valores imposibles? | 13 |
| 7 | ¿Hay valores atípicos? ¿Qué hacemos con ellos? | 14 |
| 8 | ¿Cómo se relacionan las variables? | 15–16 |
| 9 | ¿Qué aparece al mirar varias variables y crear nuevas? | 17 |
| 10 | Bitácora: «Observé…, revisaría…, decidiría… porque…» | 19 |

**Cómo trabajar**
- Las celdas marcadas con `# TODO` son para ti: reemplaza los `...` por tu código.
- Después de varios ejercicios hay una celda **✅ Comprobación**. Si no aparece un error, vas bien.
- Las celdas **✍️ Tu respuesta** son de texto: escribe tu interpretación con tus palabras. En EDA, interpretar es tan importante como calcular.

## 0. Preparación

Ejecuta esta celda para importar las librerías y cargar los datos.

**Sobre la base de datos.** Usamos la versión didáctica de *California Housing* preparada por Aurélien Géron (libro *Hands-On Machine Learning*). Procede del censo de California de 1990. **Cada fila describe un grupo de bloques censales** (una zona pequeña), no una vivienda.

El autor hizo dos modificaciones con fines didácticos: añadió la columna categórica `ocean_proximity` y retiró deliberadamente 207 valores de `total_bedrooms`. La versión que trae scikit-learn es distinta (otros nombres, otras unidades y sin esos faltantes): no las mezcles.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

URL = "https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.csv"
try:
    df = pd.read_csv("housing.csv")          # si descargaste el archivo junto al cuaderno
except FileNotFoundError:
    df = pd.read_csv(URL)                    # si no, se descarga desde GitHub

print("Datos cargados:", df.shape)

## 1. Observaciones, variables y diccionario  *(diapositivas 3 y 18)*

> **Definición (GeeksforGeeks).** Un *dataset* es una colección estructurada de datos relacionados, organizada en filas y columnas. La **variable objetivo** es la que queremos predecir o explicar usando las demás.

Antes de calcular nada, necesitamos saber qué representa una fila y qué significa cada columna. Este es el **diccionario de datos**:

| Columna | Significado | Unidad / agregación |
|---|---|---|
| `longitude`, `latitude` | Ubicación aproximada de la zona | Grados (precisión de 0,01°) |
| `housing_median_age` | Edad mediana de las viviendas de la zona | Años (el registro se corta en 52) |
| `total_rooms` | Total de habitaciones en la zona | Conteo (suma de la zona) |
| `total_bedrooms` | Total de dormitorios en la zona | Conteo (suma de la zona) |
| `population` | Personas que viven en la zona | Conteo |
| `households` | Hogares en la zona | Conteo |
| `median_income` | Ingreso mediano de los hogares | Decenas de miles de USD (aprox.) |
| `median_house_value` | Valor mediano de la vivienda (**objetivo**) | USD (el registro se corta en 500.001) |
| `ocean_proximity` | Cercanía al océano | Categoría (texto) |

In [ ]:
# TODO 1.1 · Muestra las primeras 5 filas de df
...

In [ ]:
# TODO 1.2 · Guarda en n el número de filas y en p el número de columnas (pista: df.shape)
n, p = ...
print(f"n = {n} observaciones, p = {p} columnas")

In [ ]:
# TODO 1.3 · Revisa el tipo de almacenamiento de cada columna (pista: .dtypes o .info())
...

In [ ]:
# ✅ Comprobación
assert (n, p) == (20640, 10), "Revisa n y p: usa df.shape"
print("¡Bien! La base tiene", n, "zonas y", p, "columnas.")

**✍️ Tu respuesta 1.4.** En una frase: ¿qué representa una fila de esta tabla? ¿Cuál es la variable objetivo y por qué no debe contarse como una entrada más?

*Escribe aquí…*

## 2. Naturaleza de las variables  *(diapositiva 4)*

> **Definiciones (GeeksforGeeks).** Los datos **cuantitativos** representan valores numéricos: son **discretos** si se cuentan con números enteros y **continuos** si pueden tomar cualquier valor dentro de un rango. Los datos **categóricos** clasifican en categorías: son **nominales** si no se pueden ordenar y **ordinales** si se pueden ordenar, aunque la distancia entre categorías no sea necesariamente igual.

⚠️ Recuerda: **el tipo de almacenamiento de Python (`int64`, `float64`, `object`) no define el significado estadístico.** Aquí casi todo se guardó como `float64`, pero `total_rooms` es un conteo.

In [ ]:
# TODO 2.1 · Clasifica cada columna. Usa uno de estos textos:
#   "cuantitativa discreta", "cuantitativa continua", "categórica nominal", "categórica ordinal"
tipos = {
    "longitude":          ...,
    "latitude":           ...,
    "housing_median_age": ...,
    "total_rooms":        ...,
    "total_bedrooms":     ...,
    "population":         ...,
    "households":         ...,
    "median_income":      ...,
    "median_house_value": ...,
    "ocean_proximity":    ...,
}
pd.Series(tipos, name="tipo estadístico")

In [ ]:
# ✅ Comprobación
assert tipos["ocean_proximity"] == "categórica nominal", "ocean_proximity es texto sin un orden numérico definido"
assert tipos["population"] == "cuantitativa discreta", "population es un conteo de personas"
assert tipos["median_income"] == "cuantitativa continua"
print("¡Bien clasificado!")

**Una variable ordinal.** La base no trae ninguna, así que vamos a construir una a partir del ingreso: `income_cat` con las categorías *baja < media < alta*. Esto también nos servirá en la sección 8.

In [ ]:
# TODO 2.2 · Crea income_cat con pd.cut usando los cortes [0, 2.5, 4.5, np.inf]
#           y las etiquetas ["baja", "media", "alta"]
df["income_cat"] = pd.cut(df["median_income"], bins=..., labels=...)
df["income_cat"].value_counts().sort_index()

In [ ]:
# ✅ Comprobación: pd.cut crea una categoría ORDENADA
assert df["income_cat"].cat.ordered, "income_cat debe ser ordinal"
print(df["income_cat"].cat.categories.tolist(), "→ ordenada:", df["income_cat"].cat.ordered)

## 3. Tamaño y dimensionalidad  *(diapositivas 6 y 7)*

> **Definición (GeeksforGeeks).** La *maldición de la dimensionalidad* describe cómo la eficiencia y la eficacia de los algoritmos se deterioran al aumentar la dimensionalidad: los puntos quedan dispersos y cuesta encontrar patrones.

En una tabla, **n** son las observaciones y **p** las variables consideradas. Para un modelo, *p* son sus **entradas**: el identificador y la variable objetivo no cuentan automáticamente.

In [ ]:
# TODO 3.1 · Crea la lista 'entradas' con todas las columnas originales EXCEPTO
#           el objetivo (median_house_value) y la columna que creamos (income_cat)
entradas = [c for c in df.columns if c not in [..., ...]]
print(len(entradas), "entradas posibles:", entradas)

In [ ]:
# ✅ Comprobación
assert len(entradas) == 9 and "median_house_value" not in entradas
print("¡Bien! 9 entradas posibles.")

**¿Más columnas = más información?** Altura en metros y en centímetros es la misma información. Veamos si aquí hay columnas que dicen casi lo mismo.

In [ ]:
# TODO 3.2 · Calcula la matriz de correlación (Pearson) entre estas cuatro columnas de conteo
conteos = ["total_rooms", "total_bedrooms", "population", "households"]
df[conteos]....round(2)

**✍️ Tu respuesta 3.3.** ¿Qué observas en esas correlaciones? ¿Por qué tiene sentido que zonas con más hogares tengan más dormitorios?

*Escribe aquí…*

**Experimento de cobertura (como la cuadrícula de la diapositiva 7, pero con datos reales).** Dividimos cada variable en 10 intervalos con igual número de zonas y contamos cuántas «celdas» de la cuadrícula tienen al menos una zona. Las 20.640 filas son las mismas en todos los casos; solo cambia el número de variables.

In [ ]:
variables = ["median_income", "housing_median_age", "total_rooms", "population", "households", "latitude"]

def cobertura(datos, columnas, intervalos=10):
    # Asigna a cada fila el número de intervalo (0 a 9) de cada columna
    celdas = pd.DataFrame({c: pd.qcut(datos[c], intervalos, labels=False, duplicates="drop") for c in columnas})
    ocupadas = len(celdas.drop_duplicates())        # combinaciones distintas que aparecen
    posibles = intervalos ** len(columnas)
    return ocupadas, posibles

# TODO 3.4 · Completa el ciclo para k = 1, ..., 6 variables y guarda el % de celdas ocupadas
filas = []
for k in range(1, 7):
    ocupadas, posibles = cobertura(df, ...)          # pista: las primeras k variables
    filas.append({"variables": k, "celdas posibles": posibles,
                  "celdas con datos": ocupadas, "% cubierto": ...})
tabla_cobertura = pd.DataFrame(filas)
tabla_cobertura

In [ ]:
# ✅ Comprobación
assert tabla_cobertura["% cubierto"].iloc[0] == 100, "Con 1 variable deberían cubrirse las 10 celdas"
assert tabla_cobertura["% cubierto"].iloc[-1] < 10
print("Con las mismas 20.640 filas, la cobertura cae de",
      tabla_cobertura['% cubierto'].iloc[0], "% a", tabla_cobertura['% cubierto'].iloc[-1], "%")

## 4. Análisis univariado  *(diapositivas 9 a 11)*

> **Definiciones (GeeksforGeeks).** El análisis univariado estudia **una variable a la vez**. La **media** es la suma de los valores dividida entre su número; la **mediana**, el valor del medio con los datos ordenados; la **moda**, el valor más frecuente; la **desviación estándar** mide cuánto se alejan los valores de la media; los **cuartiles** son tres puntos que dividen los datos ordenados en cuatro partes iguales. La **asimetría** positiva tiene la cola hacia la derecha y la media suele superar a la mediana.

El resumen depende del tipo de variable: **cantidades** → distribución, centro y dispersión; **categorías** → conteos, porcentajes y moda.

In [ ]:
# TODO 4.1 · Resumen numérico de las columnas cuantitativas (pista: .describe())
...

In [ ]:
# TODO 4.2 · Calcula la media y la mediana del valor de vivienda
media   = ...
mediana = ...
print(f"Media: {media:,.0f} USD | Mediana: {mediana:,.0f} USD")

In [ ]:
# TODO 4.3 · Dibuja un histograma de median_house_value con 50 intervalos
#           y marca la media y la mediana con líneas verticales (plt.axvline)
plt.hist(..., bins=50, color="#6A9FD2", edgecolor="white")
plt.axvline(media, color="#C86210", label="media")
plt.axvline(..., color="#3F8025", linestyle="--", label="mediana")
plt.xlabel("median_house_value (USD)"); plt.ylabel("Número de zonas"); plt.legend()
plt.show()

In [ ]:
# TODO 4.4 · ¿Cuántas zonas tienen el valor máximo registrado (500.001)? ¿Y cuántas la edad 52?
en_tope_valor = ...
en_tope_edad  = ...
print("Zonas en el tope de valor:", en_tope_valor, "| Zonas con edad 52:", en_tope_edad)

In [ ]:
# ✅ Comprobación
assert media > mediana, "En una distribución con cola a la derecha la media supera a la mediana"
assert en_tope_valor == 965
print("¡Correcto!")

Ahora una **categoría**: `ocean_proximity`. No tiene sentido un histograma ni una media; usamos frecuencias.

In [ ]:
# TODO 4.5 · Conteos y porcentajes de ocean_proximity (pista: value_counts(normalize=True))
conteos_op = ...
porcentajes_op = ...
pd.DataFrame({"zonas": conteos_op, "%": (100 * porcentajes_op).round(1)})

In [ ]:
# TODO 4.6 · Gráfico de barras horizontales de conteos_op
conteos_op.plot(kind=..., color="#C86210")
plt.xlabel("Número de zonas"); plt.gca().invert_yaxis()
plt.show()

**✍️ Tu respuesta 4.7.** Describe la forma de la distribución del valor de vivienda (concentración, asimetría, posibles límites de registro). ¿Qué te llama la atención de `ISLAND`?

*Escribe aquí…*

## 5. Datos faltantes  *(diapositiva 12)*

> **Definiciones (GeeksforGeeks).** Los valores faltantes aparecen cuando algunas entradas de un dataset quedan en blanco (se representan como `NaN` o `None`); pueden deberse a problemas técnicos, errores humanos, privacidad o errores de procesamiento. La **imputación** completa los faltantes con valores estimados, y puede introducir sesgo si las suposiciones son incorrectas. `isna()` devuelve `True` donde falta el dato.

Un dato faltante **no es un cero**.

In [ ]:
# TODO 5.1 · Cuenta los faltantes por columna y su porcentaje
faltantes = ...
pct_faltantes = (100 * faltantes / len(df)).round(2)
pd.DataFrame({"faltantes": faltantes, "%": pct_faltantes}).query("faltantes > 0")

**¿Por qué no rellenar con 0?** Comparemos la media de `total_bedrooms` sin los faltantes y «rellenando» con cero.

In [ ]:
# TODO 5.2 · Calcula ambas medias
media_sin_faltantes = df["total_bedrooms"].mean()          # pandas ignora NaN
media_con_ceros     = ...                                   # pista: .fillna(0).mean()
print(f"Sin faltantes: {media_sin_faltantes:.1f} | Rellenando con 0: {media_con_ceros:.1f}")

**Decisión: conservar y señalar.** En vez de borrar filas o inventar valores, creamos una columna que marque dónde faltaba el dato. Así no perdemos información y la decisión queda visible.

In [ ]:
# TODO 5.3 · Crea la columna booleana 'bedrooms_faltante' (True donde total_bedrooms es NaN)
df["bedrooms_faltante"] = ...
df["bedrooms_faltante"].sum()

In [ ]:
# ✅ Comprobación
assert faltantes["total_bedrooms"] == 207 and faltantes.drop(["total_bedrooms"]).sum() == 0
assert media_con_ceros < media_sin_faltantes
assert df["bedrooms_faltante"].sum() == 207
print("¡Bien! 207 faltantes señalados. Rellenar con 0 habría bajado la media.")

> 📌 **Si después se construye un modelo:** primero se separa un conjunto de prueba y **solo después** se aprende cualquier regla de imputación (por ejemplo, la mediana) con los datos de entrenamiento. Aprenderla con todos los datos filtra información de prueba al modelo (*data leakage*). Ese procedimiento se trabaja en la sesión de validación.

## 6. Duplicados y coherencia  *(diapositiva 13)*

> **Definición (GeeksforGeeks).** `duplicated()` devuelve `True` para las filas repetidas y `False` para las únicas; con `subset` se indican las columnas que definen la repetición.

La definición de «duplicado» depende de **qué representa una fila**. Esta base no tiene identificador de zona.

In [ ]:
# TODO 6.1 · ¿Cuántas filas están completamente repetidas?
dup_completas = ...
# TODO 6.2 · ¿Cuántas filas repiten las mismas coordenadas (longitude, latitude)?
dup_coordenadas = ...
print("Filas idénticas:", dup_completas, "| Coordenadas repetidas:", dup_coordenadas)

**✍️ Tu respuesta 6.3.** Hay miles de coordenadas repetidas. ¿Significa eso que hay miles de duplicados que debemos eliminar? Pista: mira la precisión de las coordenadas en el diccionario.

*Escribe aquí…*

**Coherencia:** ¿hay valores imposibles según su definición?

In [ ]:
# TODO 6.4 · Cuenta las zonas donde:
#   a) hay más dormitorios que habitaciones
#   b) hay más hogares que personas
#   c) alguna columna de conteo es negativa
mas_dorm_que_hab = ...
mas_hogares_que_personas = ...
conteos_negativos = (df[["total_rooms", "total_bedrooms", "population", "households"]] < 0).sum().sum()
print(mas_dorm_que_hab, mas_hogares_que_personas, conteos_negativos)

In [ ]:
# TODO 6.5 · Muestra las zonas con más hogares que personas para revisarlas
df[...][["population", "households", "total_rooms", "median_income", "median_house_value"]]

In [ ]:
# ✅ Comprobación
assert dup_completas == 0 and mas_hogares_que_personas == 3
print("Bien: 0 filas idénticas y 3 zonas incoherentes que conviene revisar.")

## 7. Valores atípicos  *(diapositiva 14)*

> **Definiciones (GeeksforGeeks).** Un valor atípico es una observación que se aparta del patrón general de los datos. El rango intercuartílico es **IQR = Q3 − Q1**; los puntos por debajo de **Q1 − 1,5·IQR** o por encima de **Q3 + 1,5·IQR** se señalan.

La regla **señala candidatos a revisar**; no prueba que el dato sea un error ni obliga a eliminarlo. Orden: observar → señalar → revisar el contexto → decidir con una razón.

In [ ]:
# TODO 7.1 · Completa la función que devuelve los límites inferior y superior según la regla IQR
def limites_iqr(serie):
    q1 = serie.quantile(0.25)
    q3 = ...
    iqr = ...
    return q1 - 1.5 * iqr, ...

li, ls = limites_iqr(df["median_income"])
print(f"Límites para median_income: {li:.2f} y {ls:.2f}")

In [ ]:
# TODO 7.2 · Para cada columna, cuenta cuántas zonas quedan fuera de los límites
resumen = []
for col in ["median_income", "population", "total_rooms", "housing_median_age"]:
    li, ls = limites_iqr(df[col])
    fuera = ...
    resumen.append({"columna": col, "límite inferior": round(li, 2), "límite superior": round(ls, 2), "señaladas": fuera})
pd.DataFrame(resumen)

In [ ]:
# TODO 7.3 · Diagrama de caja de population (pista: plt.boxplot(...))
plt.boxplot(df["population"])
plt.ylabel("population (personas por zona)")
plt.show()
# TODO 7.4 · Muestra las 5 zonas con mayor población
df....(5, "population")[["population", "households", "total_rooms", "median_income", "ocean_proximity"]]

In [ ]:
# ✅ Comprobación
li_t, ls_t = limites_iqr(df["median_income"])
assert round(ls_t, 2) == 8.01, "Revisa q3 e iqr en limites_iqr"
print("¡Bien! La regla IQR está bien implementada.")

**✍️ Tu respuesta 7.5.** ¿Las zonas con poblaciones enormes son errores? Mira sus hogares y habitaciones. ¿Qué harías con ellas y por qué?

*Escribe aquí…*

## 8. Relaciones entre variables  *(diapositivas 15 y 16)*

> **Definiciones (GeeksforGeeks).** La **correlación** describe la fuerza y la dirección de la relación entre dos variables (de −1 a 1). **Pearson** mide la relación **lineal** entre dos variables continuas. **Spearman** mide la relación entre variables **ordenadas por rangos**. **Kendall** compara el orden de los pares: **concordantes** (suben o bajan juntos) y **discordantes** (uno sube y el otro baja). **Correlación no implica causalidad.**

Los tipos de variables orientan la comparación: dos cantidades → dispersión; cantidad + categoría → cajas por grupo; dos categorías → tabla de contingencia.

In [ ]:
# TODO 8.1 · Gráfico de dispersión: median_income (x) contra median_house_value (y)
plt.scatter(..., ..., s=3, alpha=0.2, color="#1F4E79")
plt.xlabel("median_income (decenas de miles USD)"); plt.ylabel("median_house_value (USD)")
plt.show()

In [ ]:
# TODO 8.2 · Calcula los tres coeficientes entre median_income y median_house_value
#           (pista: df["a"].corr(df["b"], method="pearson" | "spearman" | "kendall"))
x, y = df["median_income"], df["median_house_value"]
coef = {
    "Pearson":  ...,
    "Spearman": ...,
    "Kendall":  ...,
}
pd.Series(coef).round(3)

In [ ]:
# ✅ Comprobación
assert 0.68 < coef["Pearson"] < 0.70
print("¡Correcto! Asociación positiva moderada-fuerte.")

**¿Una correlación cercana a 0 significa «sin relación»?** Ejecuta este ejemplo sintético (no son datos de California).

In [ ]:
rng = np.random.default_rng(0)
u = np.linspace(-5, 5, 60)
v = u**2 + rng.normal(0, 1.5, 60)            # relación en forma de U
s = pd.Series(v)
print("Pearson:", round(pd.Series(u).corr(s), 2), "| Spearman:", round(pd.Series(u).corr(s, method="spearman"), 2))
plt.scatter(u, v, color="#1F4E79"); plt.title("Relación clara, correlación ≈ 0"); plt.show()

In [ ]:
# TODO 8.3 · Cantidad + categoría: diagrama de cajas del valor por ocean_proximity
#           (pista: df.boxplot(column=..., by=...))
df.boxplot(column=..., by=..., grid=False, rot=20)
plt.suptitle(""); plt.title("Valor de vivienda por cercanía al océano"); plt.ylabel("USD")
plt.show()

In [ ]:
# TODO 8.4 · Categoría + categoría: tabla de contingencia ocean_proximity × income_cat
#           con porcentajes por fila (pista: pd.crosstab(..., normalize="index"))
tabla = pd.crosstab(..., ..., normalize=...)
(100 * tabla).round(1)

**✍️ Tu respuesta 8.5.** ¿Qué relación observas entre ingreso y valor? ¿Qué **no** podemos concluir? ¿Qué ves en la parte superior del gráfico de dispersión?

*Escribe aquí…*

## 9. Varias variables y nuevas variables  *(diapositiva 17)*

> **Definiciones (GeeksforGeeks).** El análisis **multivariado** estudia tres o más variables a la vez. La **ingeniería de variables** consiste en seleccionar, crear o modificar variables para que los patrones se aprendan mejor; la creación de variables se apoya en el conocimiento del dominio o en patrones observados.

Recuerda la sección 3: los conteos totales miden sobre todo el **tamaño** de la zona. Dividir por el número de hogares describe mejor **cómo son** las viviendas.

In [ ]:
# TODO 9.1 · Antes de dividir: ¿hay hogares o habitaciones iguales a 0? (denominadores)
print("households == 0:", ..., "| total_rooms == 0:", ...)

In [ ]:
# TODO 9.2 · Crea tres variables nuevas
df["rooms_per_household"]      = ...   # habitaciones por hogar
df["bedrooms_per_room"]        = ...   # dormitorios / habitaciones
df["population_per_household"] = ...   # personas por hogar
df[["rooms_per_household", "bedrooms_per_room", "population_per_household"]].describe().round(2)

In [ ]:
# TODO 9.3 · Compara la correlación (Spearman) con el objetivo de las variables originales y las nuevas
comparar = ["total_rooms", "total_bedrooms", "rooms_per_household", "bedrooms_per_room", "median_income"]
df[comparar + ["median_house_value"]].corr(method=...)["median_house_value"].drop("median_house_value").sort_values()

In [ ]:
# ✅ Comprobación
assert abs(df["rooms_per_household"].iloc[0] - 880 / 126) < 1e-9
print("¡Bien! Variables creadas correctamente.")

**Añadir una tercera variable al gráfico.** Ubicamos cada zona en el mapa y la coloreamos según `ocean_proximity`. ¿El patrón de valor cambia según la ubicación?

In [ ]:
# TODO 9.4 · Dispersión longitude–latitude coloreada por ocean_proximity
colores = {"<1H OCEAN": "#1F4E79", "INLAND": "#C86210", "NEAR OCEAN": "#3F8025", "NEAR BAY": "#6A9FD2", "ISLAND": "#B0302A"}
for categoria, grupo in df.groupby("ocean_proximity"):
    plt.scatter(grupo[...], grupo[...], s=2, alpha=0.4, color=colores[categoria], label=categoria)
plt.xlabel("longitude"); plt.ylabel("latitude"); plt.legend(markerscale=6)
plt.show()

In [ ]:
# TODO 9.5 · ¿La relación ingreso–valor se mantiene dentro de cada grupo?
#           Calcula la correlación de Spearman entre median_income y median_house_value POR grupo
por_grupo = df.groupby("ocean_proximity").apply(
    lambda g: g["median_income"].corr(g["median_house_value"], method=...), include_groups=False)
por_grupo.round(2)

**✍️ Tu respuesta 9.6.** ¿`rooms_per_household` o `bedrooms_per_room` aportan algo que los totales no aportaban? ¿La relación ingreso–valor se mantiene en cada grupo? ¿Por qué hay que mirar `ISLAND` con cuidado?

*Escribe aquí…*

## 10. Bitácora de decisiones  *(diapositiva 19)*

Una buena decisión de calidad conecta un **hallazgo** con una **acción** y una **razón**. Completa la bitácora con **al menos tres** hallazgos de este cuaderno usando la estructura:

> «**Observé** …, **revisaría** …, **decidiría** … **porque** …»

Ejemplo: *Observé que 965 zonas tienen `median_house_value` = 500.001; revisaría si ese valor es un tope de registro en la fuente; decidiría señalarlas y no tratarlas como valores exactos, porque el valor real puede ser mayor y sesgaría los resúmenes.*

In [ ]:
# TODO 10.1 · Completa tu bitácora (agrega las filas que necesites)
bitacora = pd.DataFrame([
    {"observé": "...", "revisaría": "...", "decidiría": "...", "porque": "..."},
    {"observé": "...", "revisaría": "...", "decidiría": "...", "porque": "..."},
    {"observé": "...", "revisaría": "...", "decidiría": "...", "porque": "..."},
])
pd.set_option("display.max_colwidth", None)
bitacora

---
### Para cerrar
- El EDA **no termina con un modelo**: termina con preguntas mejor planteadas y decisiones justificadas.
- Si estos datos se usan después para un modelo, **reserva primero el conjunto de prueba** y aprende cualquier transformación (imputación, escalamiento, codificación) solo con entrenamiento. Si el modelo se aplicará a regiones nuevas, conviene que la evaluación respete la dependencia espacial.
- Escalamiento, codificación y reducción de dimensionalidad se trabajan en las próximas sesiones.

**Fuentes de las definiciones:** GeeksforGeeks — *Exploratory Data Analysis*; *Data Types in Statistics*; *Curse of Dimensionality in Machine Learning*; *Measures of Central Tendency and Dispersion*; *Skewness of Statistical Data*; *Handling Missing Values in Machine Learning*; *What is Data Imputation*; *Pandas DataFrame duplicated()*; *Interquartile Range to Detect Outliers in Data*; *Correlation and Regression*; *Kendall Correlation Testing*; *What is Feature Engineering*.
**Datos:** Géron, A. *Hands-On Machine Learning*, versión didáctica de California Housing (github.com/ageron/handson-ml).
**Marco del EDA:** Ghosh, A. et al. (2018). *A comprehensive review of tools for exploratory analysis of tabular industrial datasets*. Visual Informatics, 2, 235–253.